In [2]:
import os

import logging
import time
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import csv
import pandas as pd 
import numpy as np 

from eutils import EutilsNCBIError, EutilsRequestError
from metapub import PubMedFetcher, pubmedcentral
from datetime import datetime

from Functions import DataRetrieval as DR

#API_KEY
from Reference_files.keys import API_KEY as API_KEY

In [3]:
# Initialize logger
prefix = "test"+str(datetime.now()).split()[0]
file_handler = logging.FileHandler(f"{prefix}_Examples.log", mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

# From Query to pubmedCentral full text

### the following pipeline starts with a file with the query which is in the folder Reference files to PMC full text
#### pubmed central 

In [4]:
# Initialize PubMed fetcher
fetcher = PubMedFetcher()

In [5]:
query_file = "./Reference_files/query.txt"
query = DR.read_query_from_file(query_file)

In [7]:
## Fetch openaccess PMCs from a query
start_date = "2000-01-01"
stop_date = None  # Will default to the current date if None
pmid_array = DR.fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)
pmc_id_list = DR.get_pmcid_for_otherid(pmid_array)
oa_file_list = "Reference_files/oa_file_list.csv"
oa_pmcs = DR.filter_oa_database(oa_file_list, pmc_id_list)

2025-03-31 13:42:34 LAPTOP-S8N3C7A8 root[11672] INFO Total PMIDs fetched: 100


In [ ]:
def main():
    # Step 1: Read the query from a file
    query_file = "./Reference_files/query.txt" 
    query = DR.read_query_from_file(query_file)
    if not query:
        logging.error("Query reading failed. Exiting.")
        return
    # Step 2: Fetch PMIDs over a specified period
    start_date = "2000-01-01"
    stop_date = None  # Will default to the current date if None
    pmid_array = DR.fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)
    if pmid_array.size == 0:
        logging.error("No PMIDs fetched. Exiting.")
        return
    # Step 3: Retrieve PMCIDs for the fetched PMIDs
    pmc_id_list = DR.get_pmcid_for_otherid(pmid_array)
    # Step 5: Filter the OA database using the retrieved PMCIDs
    oa_file_list = "oa_file_list.csv"  # OA file list CSV filename
    oa_pmcs = DR.filter_oa_database(oa_file_list, pmc_id_list)

    logging.info("OA database filtering completed.")
    return oa_pmcs

# Call the main function to execute the workflow
if __name__ == "__main__":
    main()

In [ ]:
## Download papers using python requests
import requests

# Create directory
os.makedirs('./Full_text_jsons', exist_ok=True)

for pmcid in oa_pmcids:
    url = f'https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode'
    response = requests.get(url)
    if response.status_code == 200:
        with open(f'./Full_text_jsons/{pmcid}.json', 'w') as f:
            f.write(response.text)
    else:
        print(f"Failed to download {pmcid}")

In [ ]:
## Download using powershell \faster\ 
# Save the oa_pmcids pandas series to a text file that powershell can read(one ID per line)
oa_pmcids.to_csv('oa_pmcids.txt', index=False, header=False)

!powershell -Command "mkdir -p ./Full_text_jsons; Get-Content oa_pmcids.txt | ForEach-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/' + $_ + '/unicode') -OutFile ('./Full_text_jsons/' + $_ + '.json') }"

In [ ]:
!curl -s "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC10759277/unicode" > "PMC10759277.json"


In [ ]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${url}" > "./Full_text_jsons/${PMCID}.json"
done < full_text_pmc.txt



In [ ]:
!powershell -Command "mkdir -p ./Full_text_jsons; Get-Content full_text_pmc.txt | ForEach-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/' + $_ + '/unicode') -OutFile ('./Full_text_jsons/' + $_ + '.json') }"

# Publisher Metadata

# Paper Metadata

In [ ]:

pmids = ["12345678", "23456789"]  # Example PMIDs
try1 = DR.fetch_articles_to_dataframe(pmids)